# Coral Reef Health Detector (Client Version)

This Google Colab Notebook allows you to run high-accuracy Transformer (RT-DETR) AI detection on coral images or videos without needing any heavy hardware on your laptop. **Google provides the computing power for free.**

### Step 1: Install AI Packages
Run this cell to install the necessary detection framework on the Google Server.

In [ ]:
!pip install ultralytics opencv-python-headless
import cv2
from ultralytics import RTDETR
from google.colab import files
from IPython.display import display, Image
import os

### Step 2: Upload Files
Run this cell to select the image (e.g., .jpg) or video (e.g., .mp4) you want to analyze. The system assumes your `rtdetr_best.pt` model is already placed in `/content/`.

In [ ]:
print("Please select your test image/video sequence now:")
uploaded = files.upload()
test_file = list(uploaded.keys())[0] if uploaded else None
if not test_file:
    print("ERROR: You forgot to upload a test image or video!")

### Step 3: Run Analysis
Run this cell to load the heavy AI model and scan your coral file.

In [ ]:
import os
from ultralytics import RTDETR
from IPython.display import display, Image
from google.colab import files

model_path = '/content/rtdetr_best.pt'

if not os.path.exists(model_path):
    print(f"ERROR: I can't find your model at {model_path}! Please check the left sidebar.")
elif not test_file or not os.path.exists(test_file):
    print("ERROR: I can't find your test image or video! Please run Step 2 again to upload one.")
else:
    print(f"Analyzing '{test_file}' using your model '{model_path}'...\n")
    
    model = RTDETR(model_path)
    results = model.predict(source=test_file, save=True, project="results", name="predict")
    
    print("\n" + "="*40)
    print("DETECTION SUMMARY:")
    print("="*40)
    # Print the exact text percentages for the user to see under the cell
    for box in results[0].boxes:
        class_id = int(box.cls[0])
        class_name = model.names[class_id]
        confidence = float(box.conf[0]) * 100
        if class_name == "Healthy Coral":
            print(f"[Healthy]     Healthy Coral: {confidence:.2f}%")
        elif "disease" in class_name.lower():
            print(f"[Disease]     {class_name}: {confidence:.2f}%")
        elif "dead" in class_name.lower() or "bleached" in class_name.lower() or "pox" in class_name.lower():
            print(f"[Warning]     {class_name}: {confidence:.2f}%")
        else:
            print(f"[Detected]    {class_name}: {confidence:.2f}%")
    print("="*40 + "\n")

    # Safely get the exact output directory where ultralytics saved the photo
    save_dir = results[0].save_dir
    final_result_path = os.path.join(save_dir, test_file)
    
    if os.path.exists(final_result_path):
        if final_result_path.lower().endswith(('.png', '.jpg', '.jpeg')):
            display(Image(filename=final_result_path))
        
        print(f"\nPrompting browser download of your annotated file...")
        files.download(final_result_path)
    else:
        print("Output file missing. Something went wrong with the save!")